In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split

# ── PATHS — sesuaikan dengan environment kamu ──
TRAIN_PATH = r'C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\train.csv'
TEST_PATH  = r'C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\test.csv'
SAMPLE_PATH = r'C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\sample submission.csv'
OUTPUT_PATH = r'C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\submission.csv'

# ── CONFIG ──
RANDOM_SEED   = 42
YEAR_CUTOFF   = 1990     # hanya pakai data >= tahun ini untuk training
VAL_YEAR_CUT  = 2008     # validation: 2008–2011, train: 1990–2007

# ELO config
ELO_K_DEFAULT = 20
ELO_K_WC      = 40
ELO_INIT      = 1500.0
ELO_WC_TOURNAMENTS = ['FIFA World Cup', 'AFC Asian Cup', 'UEFA Euro',
                       'Africa Cup of Nations', 'Copa América', 'Gold Cup']

print('Imports OK')

Imports OK


In [2]:
train_raw = pd.read_csv(TRAIN_PATH, parse_dates=['date'])
test_raw  = pd.read_csv(TEST_PATH,  parse_dates=['date'])
sample    = pd.read_csv(SAMPLE_PATH)

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'Train date: {train_raw["date"].min().date()} → {train_raw["date"].max().date()}')
print(f'Test  date: {test_raw["date"].min().date()} → {test_raw["date"].max().date()}')
print(f'\nKolom HANYA di train (tidak di test):')
train_only = [c for c in train_raw.columns if c not in test_raw.columns]
print(train_only)

Train: (78772, 47)  |  Test: (42422, 20)
Train date: 1872-11-30 → 2011-08-04
Test  date: 2011-08-06 → 2026-03-31

Kolom HANYA di train (tidak di test):
['team_goals', 'opp_goals', 'team_points_last5', 'opp_points_last5', 'points_last5_diff', 'team_gd_last5', 'opp_gd_last5', 'gd_last5_diff', 'h2h_points_last5', 'h2h_gd_last5', 'days_since_last_match_team', 'days_since_last_match_opp', 'team_points_last10', 'opp_points_last10', 'team_avg_goals_last5', 'team_avg_conceded_last5', 'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'team_win_rate_last10', 'opp_win_rate_last10', 'elo_team', 'elo_opponent', 'rank_team', 'rank_opponent', 'rank_diff', 'rank_missing_team', 'rank_missing_opp']


In [3]:
# Fitur yang ada di train tapi TIDAK di test — harus direkonstruksi atau di-drop
ROLLING_FEATURES_TRAIN_ONLY = [
    'team_points_last5', 'opp_points_last5', 'points_last5_diff',
    'team_gd_last5', 'opp_gd_last5', 'gd_last5_diff',
    'h2h_points_last5', 'h2h_gd_last5',
    'days_since_last_match_team', 'days_since_last_match_opp',
    'team_points_last10', 'opp_points_last10',
    'team_avg_goals_last5', 'team_avg_conceded_last5',
    'opp_avg_goals_last5', 'opp_avg_conceded_last5',
    'team_win_rate_last10', 'opp_win_rate_last10',
    'elo_team', 'elo_opponent',
    'rank_team', 'rank_opponent', 'rank_diff',
    'rank_missing_team', 'rank_missing_opp'
]

# Fitur statis yang ada di KEDUA train & test
STATIC_FEATURES = [
    'gender', 'team', 'opponent', 'is_home', 'neutral', 'tournament',
    'venue_country', 'confederation_team', 'confederation_opp',
    'population_team', 'population_opp',
    'gdp_per_capita_team', 'gdp_per_capita_opp',
    'altitude_venue', 'distance_travel_team', 'distance_travel_opp',
    'temperature_venue'
]

print('Fitur statis (tersedia di train & test):', len(STATIC_FEATURES))
print('Fitur rolling train-only (harus direkonstruksi):', len(ROLLING_FEATURES_TRAIN_ONLY))

# Cek missing rate fitur rolling di train
print('\n=== Missing rate fitur rolling di train ===')
for f in ROLLING_FEATURES_TRAIN_ONLY:
    if f in train_raw.columns:
        pct = train_raw[f].isna().mean()
        print(f'  {f}: {pct:.2%}')

Fitur statis (tersedia di train & test): 17
Fitur rolling train-only (harus direkonstruksi): 25

=== Missing rate fitur rolling di train ===
  team_points_last5: 0.37%
  opp_points_last5: 0.22%
  points_last5_diff: 0.56%
  team_gd_last5: 0.37%
  opp_gd_last5: 0.22%
  gd_last5_diff: 0.56%
  h2h_points_last5: 15.20%
  h2h_gd_last5: 15.20%
  days_since_last_match_team: 0.37%
  days_since_last_match_opp: 0.22%
  team_points_last10: 0.37%
  opp_points_last10: 0.22%
  team_avg_goals_last5: 0.37%
  team_avg_conceded_last5: 0.37%
  opp_avg_goals_last5: 0.22%
  opp_avg_conceded_last5: 0.22%
  team_win_rate_last10: 0.37%
  opp_win_rate_last10: 0.22%
  elo_team: 0.00%
  elo_opponent: 0.00%
  rank_team: 53.25%
  rank_opponent: 53.25%
  rank_diff: 55.73%
  rank_missing_team: 0.00%
  rank_missing_opp: 0.00%


In [4]:
def compute_match_outcome(team_goals, opp_goals):
    """Return points (3/1/0) untuk team."""
    if team_goals > opp_goals: return 3
    if team_goals == opp_goals: return 1
    return 0

def get_elo_k(tournament):
    """K-factor berdasarkan turnamen."""
    if tournament in ELO_WC_TOURNAMENTS:
        return ELO_K_WC
    return ELO_K_DEFAULT

def build_historical_features(train_df, test_df):
    """
    Rekonstruksi semua fitur historis secara forward-only.
    Input: train_df (dengan target), test_df (tanpa target)
    Output: dataframe gabungan dengan kolom historis yang sudah direkonstruksi
    """
    print('Building historical features...')
    
    # Ambil kolom yang dibutuhkan dari train
    train_cols = ['Id', 'match_id', 'date', 'gender', 'team', 'opponent',
                  'is_home', 'neutral', 'tournament', 'venue_country',
                  'team_goals', 'opp_goals'] + STATIC_FEATURES
    train_cols = list(dict.fromkeys(train_cols))  # deduplicate
    
    test_cols = ['Id', 'match_id', 'date', 'gender', 'team', 'opponent',
                 'is_home', 'neutral', 'tournament', 'venue_country'] + STATIC_FEATURES
    test_cols = list(dict.fromkeys(test_cols))
    
    tr = train_df[[c for c in train_cols if c in train_df.columns]].copy()
    te = test_df[[c for c in test_cols if c in test_df.columns]].copy()
    te['team_goals'] = np.nan
    te['opp_goals']  = np.nan
    
    # Gabung & sort by date
    all_data = pd.concat([tr, te], ignore_index=True).sort_values(['date', 'match_id', 'Id'])
    all_data['is_test'] = all_data['team_goals'].isna()
    
    # ── STEP 1: ELO ──────────────────────────────────────────────────────────
    print('  Computing ELO...')
    elo_dict = {}  # {(team, gender): elo_value}
    
    def get_elo(team, gender):
        return elo_dict.get((team, gender), ELO_INIT)
    
    elo_team_list = []
    elo_opp_list  = []
    
    # Process match by match (grouped by match_id + date untuk handle dua baris per match)
    # Ambil pre-match ELO dulu untuk SEMUA baris, baru update setelahnya
    processed_matches = set()
    
    for idx, row in all_data.iterrows():
        t = (row['team'], row['gender'])
        o = (row['opponent'], row['gender'])
        elo_team_list.append(get_elo(*t))
        elo_opp_list.append(get_elo(*o))
    
    all_data['elo_team_recon']     = elo_team_list
    all_data['elo_opponent_recon'] = elo_opp_list
    
    # Update ELO setelah menyimpan pre-match values
    # Proses per match_id (bukan per baris) untuk avoid double-update
    elo_dict = {}  # reset
    match_groups = all_data.dropna(subset=['team_goals']).groupby('match_id')
    
    # Rebuild ELO update secara kronologis
    elo_pre_team = {}
    elo_pre_opp  = {}
    
    for match_id, grp in all_data.groupby('match_id', sort=False):
        # Simpan pre-match ELO untuk semua baris di match ini
        for idx, row in grp.iterrows():
            t = (row['team'], row['gender'])
            o = (row['opponent'], row['gender'])
            elo_pre_team[idx] = get_elo(*t)
            elo_pre_opp[idx]  = get_elo(*o)
        
        # Update ELO hanya kalau ada hasil (train rows)
        first_row = grp.iloc[0]
        if not pd.isna(first_row['team_goals']):
            # Ambil satu pasang baris (home/away atau dua perspektif)
            # Gunakan baris pertama untuk update ELO (satu kali per match)
            row = first_row
            t = (row['team'], row['gender'])
            o = (row['opponent'], row['gender'])
            e_t = get_elo(*t)
            e_o = get_elo(*o)
            
            expected_t = 1 / (1 + 10 ** ((e_o - e_t) / 400))
            
            tg, og = row['team_goals'], row['opp_goals']
            if   tg > og: actual_t = 1.0
            elif tg == og: actual_t = 0.5
            else:          actual_t = 0.0
            
            K = get_elo_k(row['tournament'])
            new_e_t = e_t + K * (actual_t - expected_t)
            new_e_o = e_o + K * ((1 - actual_t) - (1 - expected_t))
            elo_dict[t] = new_e_t
            elo_dict[o] = new_e_o
    
    # Assign pre-match ELO yang sudah dihitung dengan benar
    all_data['elo_team_recon']     = pd.Series(elo_pre_team)
    all_data['elo_opponent_recon'] = pd.Series(elo_pre_opp)
    
    # ── STEP 2: Rolling features per tim ─────────────────────────────────────
    print('  Computing rolling features per team...')
    
    # Compute points per baris (dari perspektif team)
    def row_points(row):
        if pd.isna(row['team_goals']): return np.nan
        return compute_match_outcome(row['team_goals'], row['opp_goals'])
    
    def row_gd(row):
        if pd.isna(row['team_goals']): return np.nan
        return row['team_goals'] - row['opp_goals']
    
    all_data['_points'] = all_data.apply(row_points, axis=1)
    all_data['_gd']     = all_data.apply(row_gd, axis=1)
    
    # Sort by date untuk rolling
    all_data = all_data.sort_values(['gender', 'team', 'date', 'match_id']).reset_index(drop=True)
    
    grp = all_data.groupby(['gender', 'team'])
    
    # Rolling last 5 — gunakan shift(1) supaya current match tidak masuk window
    all_data['team_points_last5_recon']    = grp['_points'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean() * 5)  # total points
    all_data['team_avg_goals_last5_recon'] = grp['team_goals'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    all_data['team_avg_conceded_last5_recon'] = grp['opp_goals'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    all_data['team_gd_last5_recon']        = grp['_gd'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    all_data['team_win_rate_last10_recon'] = grp['_points'].transform(
        lambda x: (x.shift(1).rolling(10, min_periods=1) \
                   .apply(lambda w: (w == 3).mean(), raw=True)))
    
    # Days since last match
    all_data['days_since_last_match_recon'] = grp['date'].transform(
        lambda x: x.diff().dt.days)
    
    # Sort kembali ke urutan original (by date, match_id, Id)
    all_data = all_data.sort_values(['date', 'match_id', 'Id']).reset_index(drop=True)
    
    # ── STEP 3: Merge opponent rolling features ───────────────────────────────
    print('  Merging opponent rolling features...')
    
    # Ambil fitur tim lawan dari sudut pandang baris lain di match yang sama
    # Trick: setiap match punya dua baris — team A vs team B, dan team B vs team A
    # Jadi untuk row team=A, opponent=B: kita perlu fitur rolling team B
    # Cukup buat lookup dari team-level rolling, merge by (gender, opponent, date approx)
    
    team_rolling = all_data[[
        'Id', 'gender', 'team',
        'team_points_last5_recon', 'team_avg_goals_last5_recon',
        'team_avg_conceded_last5_recon', 'team_gd_last5_recon',
        'team_win_rate_last10_recon', 'days_since_last_match_recon',
        'elo_team_recon'
    ]].copy()
    
    # Rename untuk perspective lawan
    opp_rolling = team_rolling.rename(columns={
        'team': 'opponent',
        'team_points_last5_recon':    'opp_points_last5_recon',
        'team_avg_goals_last5_recon': 'opp_avg_goals_last5_recon',
        'team_avg_conceded_last5_recon': 'opp_avg_conceded_last5_recon',
        'team_gd_last5_recon':        'opp_gd_last5_recon',
        'team_win_rate_last10_recon': 'opp_win_rate_last10_recon',
        'days_since_last_match_recon': 'opp_days_since_last_match_recon',
        'elo_team_recon':             'elo_opponent_recon_v2'
    })
    
    # Merge via match_id — cari baris mirror
    all_data = all_data.merge(
        all_data[['Id', 'match_id', 'gender', 'team',
                  'team_points_last5_recon', 'team_avg_goals_last5_recon',
                  'team_avg_conceded_last5_recon', 'team_gd_last5_recon',
                  'team_win_rate_last10_recon', 'days_since_last_match_recon',
                  'elo_team_recon']]
        .rename(columns={
            'team': 'opponent',
            'team_points_last5_recon': 'opp_points_last5_recon',
            'team_avg_goals_last5_recon': 'opp_avg_goals_last5_recon',
            'team_avg_conceded_last5_recon': 'opp_avg_conceded_last5_recon',
            'team_gd_last5_recon': 'opp_gd_last5_recon',
            'team_win_rate_last10_recon': 'opp_win_rate_last10_recon',
            'days_since_last_match_recon': 'opp_days_since_last_match_recon',
            'elo_team_recon': 'elo_opponent_recon'
        }),
        on=['match_id', 'gender', 'opponent'],
        how='left',
        suffixes=('', '_dup')
    )
    # Drop duplicate Id column dari merge
    if 'Id_dup' in all_data.columns:
        all_data = all_data.drop(columns=['Id_dup'])
    
    # ── STEP 4: Derived features ──────────────────────────────────────────────
    all_data['elo_diff_recon']    = all_data['elo_team_recon'] - all_data['elo_opponent_recon']
    all_data['points_diff_recon'] = all_data['team_points_last5_recon'] - all_data['opp_points_last5_recon']
    all_data['gd_diff_recon']     = all_data['team_gd_last5_recon'] - all_data['opp_gd_last5_recon']
    
    print(f'  Done. Shape: {all_data.shape}')
    return all_data

print('Functions defined.')

Functions defined.


In [5]:
# ── Jalankan rekonstruksi ──
all_data = build_historical_features(train_raw, test_raw)

train_recon = all_data[~all_data['is_test']].copy()
test_recon  = all_data[all_data['is_test']].copy()

print(f'Train recon: {train_recon.shape}')
print(f'Test  recon: {test_recon.shape}')

Building historical features...
  Computing ELO...
  Computing rolling features per team...
  Merging opponent rolling features...
  Done. Shape: (121194, 43)
Train recon: (78772, 43)
Test  recon: (42422, 43)


In [6]:
def validate_reconstruction(train_recon, train_raw):
    """Bandingkan nilai rekonstruksi vs nilai asli di train."""
    checks = [
        ('elo_team', 'elo_team_recon'),
        ('elo_opponent', 'elo_opponent_recon'),
        ('team_avg_goals_last5', 'team_avg_goals_last5_recon'),
        ('team_avg_conceded_last5', 'team_avg_conceded_last5_recon'),
        ('team_points_last5', 'team_points_last5_recon'),
    ]
    
    # Merge dengan train_raw berdasarkan Id
    merged = train_recon[['Id'] + [c[1] for c in checks]].merge(
        train_raw[['Id'] + [c[0] for c in checks]], on='Id', how='inner'
    )
    
    print('=== VALIDASI REKONSTRUKSI ===')
    print(f'{"Feature original":<35} {"Feature recon":<35} {"Corr":>6}  Status')
    print('-' * 90)
    
    all_ok = True
    for orig, recon in checks:
        if orig not in merged.columns or recon not in merged.columns:
            na_str = 'N/A'
            print(f'{orig:<35} {recon:<35} {na_str:>6}  ⚠️  SKIP')
            continue
        sub = merged[[orig, recon]].dropna()
        if len(sub) < 10:
            na_str = 'N/A'
            print(f'{orig:<35} {recon:<35} {na_str:>6}  ⚠️  TOO FEW ROWS')
            continue
        corr = sub[orig].corr(sub[recon])
        status = '✅ OK' if corr > 0.95 else ('⚠️  WARN' if corr > 0.85 else '❌ BUG!')
        if corr <= 0.85: all_ok = False
        print(f'{orig:<35} {recon:<35} {corr:>6.4f}  {status}')
    
    if all_ok:
        print('\n✅ Semua rekonstruksi OK. Lanjut ke training.')
    else:
        print('\n❌ Ada rekonstruksi yang bermasalah. JANGAN lanjut training sebelum fix.')
    return all_ok

recon_ok = validate_reconstruction(train_recon, train_raw)

=== VALIDASI REKONSTRUKSI ===
Feature original                    Feature recon                         Corr  Status
------------------------------------------------------------------------------------------
elo_team                            elo_team_recon                      0.9189  ⚠️  WARN
elo_opponent                        elo_opponent_recon                  0.9192  ⚠️  WARN
team_avg_goals_last5                team_avg_goals_last5_recon          0.8428  ❌ BUG!
team_avg_conceded_last5             team_avg_conceded_last5_recon       0.8437  ❌ BUG!
team_points_last5                   team_points_last5_recon             0.8804  ⚠️  WARN

❌ Ada rekonstruksi yang bermasalah. JANGAN lanjut training sebelum fix.


In [7]:
# Tournament weight mapping (untuk AW-MAE metric awareness)
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 2.00,
    'AFC Championship': 1.80, 'AFC Asian Cup': 1.80,
    'UEFA Euro': 1.80, 'Copa América': 1.80,
    'Africa Cup of Nations': 1.80, 'African Cup of Nations': 1.80,
    'Gold Cup': 1.75, 'CONCACAF Gold Cup': 1.75,
    'FIFA World Cup qualification': 1.50,
    'UEFA Euro qualification': 1.40,
    'AFC Asian Cup qualification': 1.40,
    'Friendly': 0.96,
}

def engineer_features(df):
    """Feature engineering — tambah fitur derived di sini."""
    df = df.copy()
    
    # Tanggal
    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month
    
    # Tournament weight (berguna sebagai signal konteks pertandingan)
    df['tournament_weight'] = df['tournament'].map(TOURNAMENT_WEIGHTS).fillna(1.20)
    
    # Altitude penalty (altitude sangat tinggi punya efek ke gol)
    df['altitude_venue_clip'] = df['altitude_venue'].clip(lower=0)  # -9999 = missing
    df['high_altitude'] = (df['altitude_venue'] > 2500).astype(int)
    
    # Distance sebagai proxy kelelahan
    df['long_travel_team'] = (df['distance_travel_team'] > 5000).astype(int)
    df['long_travel_opp']  = (df['distance_travel_opp'] > 5000).astype(int)
    
    return df

train_fe = engineer_features(train_recon)
test_fe  = engineer_features(test_recon)

print('Feature engineering done.')
print(f'Train: {train_fe.shape}, Test: {test_fe.shape}')

Feature engineering done.
Train: (78772, 50), Test: (42422, 50)


In [8]:
# ── Definisi fitur yang dipakai untuk training ──
# Eksperimen: ubah list ini secara inkremental

# Fitur statis (selalu tersedia di train & test)
CAT_FEATURES = ['gender', 'team', 'opponent', 'tournament', 'venue_country',
                'confederation_team', 'confederation_opp']

NUM_FEATURES_STATIC = [
    'is_home', 'neutral',
    'population_team', 'population_opp',
    'gdp_per_capita_team', 'gdp_per_capita_opp',
    'altitude_venue_clip', 'high_altitude',
    'distance_travel_team', 'distance_travel_opp',
    'long_travel_team', 'long_travel_opp',
    'temperature_venue',
    'tournament_weight',
    'year', 'month'
]

# Fitur historis hasil rekonstruksi
# ── EKSPERIMEN: tambah/hapus dari sini secara inkremental ──
NUM_FEATURES_HISTORICAL = [
    # ELO (Eksperimen 2 — paling aman)
    'elo_team_recon', 'elo_opponent_recon', 'elo_diff_recon',
    
    # Rolling goals (Eksperimen 3)
    'team_avg_goals_last5_recon', 'team_avg_conceded_last5_recon',
    'opp_avg_goals_last5_recon', 'opp_avg_conceded_last5_recon',
    
    # Form/points (Eksperimen 4)
    'team_points_last5_recon', 'opp_points_last5_recon', 'points_diff_recon',
    'team_gd_last5_recon', 'opp_gd_last5_recon', 'gd_diff_recon',
    'team_win_rate_last10_recon', 'opp_win_rate_last10_recon',
    
    # Days since last match
    'days_since_last_match_recon', 'opp_days_since_last_match_recon',
]

ALL_FEATURES = CAT_FEATURES + NUM_FEATURES_STATIC + NUM_FEATURES_HISTORICAL

# Pastikan semua fitur tersedia
missing_in_train = [f for f in ALL_FEATURES if f not in train_fe.columns]
missing_in_test  = [f for f in ALL_FEATURES if f not in test_fe.columns]
print(f'Fitur hilang di train: {missing_in_train}')
print(f'Fitur hilang di test:  {missing_in_test}')
print(f'Total fitur: {len(ALL_FEATURES)}')

Fitur hilang di train: []
Fitur hilang di test:  []
Total fitur: 40


In [9]:
def compute_awmae(df_true, team_goals_pred, opp_goals_pred, weights=None):
    """
    Hitung AW-MAE.
    df_true: DataFrame dengan kolom team_goals, opp_goals, tournament
    team_goals_pred, opp_goals_pred: array prediksi
    """
    tg_true = df_true['team_goals'].values
    og_true = df_true['opp_goals'].values
    tg_pred = np.array(team_goals_pred)
    og_pred = np.array(opp_goals_pred)
    
    # Round prediksi ke integer untuk komponen exact/outcome/gd
    tg_pred_r = np.round(tg_pred).astype(int).clip(0)
    og_pred_r = np.round(og_pred).astype(int).clip(0)
    
    # 1. Base MAE per match
    mae = (np.abs(tg_true - tg_pred) + np.abs(og_true - og_pred)) / 2
    
    # 2. Exact score
    exact = ((tg_pred_r == tg_true) & (og_pred_r == og_true)).astype(float)
    
    # 3. Outcome (W/D/L)
    true_outcome = np.sign(tg_true - og_true)
    pred_outcome = np.sign(tg_pred_r - og_pred_r)
    outcome_correct = (true_outcome == pred_outcome).astype(float)
    
    # 4. Goal difference
    true_gd = tg_true - og_true
    pred_gd = tg_pred_r - og_pred_r
    gd_correct = (true_gd == pred_gd).astype(float)
    
    # 5. Penalty
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome_correct) + 0.15 * (1 - gd_correct)
    
    # 6. Outcome multiplier
    multiplier = np.where(outcome_correct == 1, 1.0, 1.5)
    
    # 7. Non-linear scaling
    raw_loss = mae + penalty
    loss = (raw_loss * multiplier) ** 1.5
    
    # 8. Tournament weights
    if weights is None:
        w = df_true['tournament'].map(TOURNAMENT_WEIGHTS).fillna(1.20).values
    else:
        w = np.array(weights)
    
    awmae = np.sum(loss * w) / np.sum(w)
    
    # Breakdown diagnostik
    diag = {
        'AW-MAE': awmae,
        'MAE (raw)': mae.mean(),
        'Exact %': exact.mean() * 100,
        'Outcome % correct': outcome_correct.mean() * 100,
        'GD % correct': gd_correct.mean() * 100,
        'Avg penalty': penalty.mean(),
        'Avg loss': loss.mean()
    }
    return awmae, diag

print('AW-MAE metric defined.')

AW-MAE metric defined.


In [10]:
# Filter era modern
train_modern = train_fe[train_fe['year'] >= YEAR_CUTOFF].copy()
print(f'Train modern (year>={YEAR_CUTOFF}): {len(train_modern)} rows')

# Forward-looking split: train on pre-VAL_YEAR_CUT, validate on post
train_split = train_modern[train_modern['year'] <  VAL_YEAR_CUT].copy()
val_split   = train_modern[train_modern['year'] >= VAL_YEAR_CUT].copy()

print(f'Train split: {len(train_split)} rows ({train_split["year"].min()}–{train_split["year"].max()})')
print(f'Val split:   {len(val_split)}   rows ({val_split["year"].min()}–{val_split["year"].max()})')

# Cek NaN di fitur
for f in ALL_FEATURES:
    if f not in train_split.columns: continue
    pct = train_split[f].isna().mean()
    if pct > 0.05:
        print(f'  ⚠️  {f}: {pct:.2%} missing di train_split')

Train modern (year>=1990): 43610 rows
Train split: 34680 rows (1990–2007)
Val split:   8930   rows (2008–2011)
  ⚠️  population_team: 7.14% missing di train_split
  ⚠️  population_opp: 7.14% missing di train_split
  ⚠️  gdp_per_capita_team: 19.67% missing di train_split
  ⚠️  gdp_per_capita_opp: 19.67% missing di train_split
  ⚠️  altitude_venue_clip: 25.33% missing di train_split
  ⚠️  distance_travel_team: 38.41% missing di train_split
  ⚠️  distance_travel_opp: 38.41% missing di train_split
  ⚠️  temperature_venue: 14.12% missing di train_split


In [11]:
def make_pool(df, features, cat_features, target=None):
    """Buat CatBoost Pool dengan handling NaN dan cat features."""
    X = df[features].copy()
    
    # Fill NaN numerik dengan median (bukan 0 — ini penting!)
    num_features = [f for f in features if f not in cat_features]
    for f in num_features:
        if X[f].isna().any():
            X[f] = X[f].fillna(X[f].median())
    
    # Cat features — fill NaN dengan 'Unknown'
    for f in cat_features:
        if f in X.columns:
            X[f] = X[f].fillna('Unknown').astype(str)
    
    cat_idx = [features.index(f) for f in cat_features if f in features]
    
    if target is not None:
        y = df[target].values
        return Pool(X, label=y, cat_features=cat_idx)
    else:
        return Pool(X, cat_features=cat_idx)

def train_model(train_df, val_df, target, features, cat_features, verbose=100):
    """Train satu CatBoostRegressor."""
    train_pool = make_pool(train_df, features, cat_features, target)
    val_pool   = make_pool(val_df, features, cat_features, target)
    
    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        loss_function='MAE',        # MAE loss lebih robust untuk gol (banyak outlier)
        eval_metric='MAE',
        random_seed=RANDOM_SEED,
        early_stopping_rounds=100,
        verbose=verbose,
        use_best_model=True
    )
    
    model.fit(train_pool, eval_set=val_pool)
    return model

print('Training functions defined.')

Training functions defined.


In [12]:
# ── Train model validasi ──
print('=== Training model_team_goals ===')
model_tg_val = train_model(train_split, val_split, 'team_goals', ALL_FEATURES, CAT_FEATURES)

print('\n=== Training model_opp_goals ===')
model_og_val = train_model(train_split, val_split, 'opp_goals', ALL_FEATURES, CAT_FEATURES)

=== Training model_team_goals ===
0:	learn: 1.1600929	test: 1.0951615	best: 1.0951615 (0)	total: 112ms	remaining: 2m 47s
100:	learn: 1.0093380	test: 0.9699033	best: 0.9699033 (100)	total: 3.54s	remaining: 49s
200:	learn: 0.9907543	test: 0.9656883	best: 0.9656475 (198)	total: 7.03s	remaining: 45.4s
300:	learn: 0.9747241	test: 0.9638456	best: 0.9638337 (299)	total: 10.5s	remaining: 41.6s
400:	learn: 0.9611230	test: 0.9617849	best: 0.9615336 (381)	total: 13.9s	remaining: 38.2s
500:	learn: 0.9514887	test: 0.9612392	best: 0.9612026 (499)	total: 17.2s	remaining: 34.4s
600:	learn: 0.9424315	test: 0.9611658	best: 0.9608069 (518)	total: 20.6s	remaining: 30.8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9608069227
bestIteration = 518

Shrink model to first 519 iterations.

=== Training model_opp_goals ===
0:	learn: 1.1612794	test: 1.0960686	best: 1.0960686 (0)	total: 47.6ms	remaining: 1m 11s
100:	learn: 1.0101672	test: 0.9668878	best: 0.9668878 (100)	total: 3.69s	remaini

In [13]:
def evaluate(model_tg, model_og, val_df, features, cat_features, label='Validation'):
    """Evaluasi AW-MAE dan diagnostik."""
    val_pool_tg = make_pool(val_df, features, cat_features)
    val_pool_og = make_pool(val_df, features, cat_features)
    
    pred_tg = model_tg.predict(val_pool_tg).clip(0)
    pred_og = model_og.predict(val_pool_og).clip(0)
    
    awmae, diag = compute_awmae(val_df, pred_tg, pred_og)
    
    print(f'\n=== {label} Results ===')
    for k, v in diag.items():
        print(f'  {k:<25}: {v:.4f}')
    
    return pred_tg, pred_og, awmae, diag

pred_tg_val, pred_og_val, awmae_val, diag_val = evaluate(
    model_tg_val, model_og_val, val_split, ALL_FEATURES, CAT_FEATURES
)

# Dummy baseline (selalu prediksi 1-1)
dummy_tg = np.ones(len(val_split))
dummy_og = np.ones(len(val_split))
awmae_dummy, _ = compute_awmae(val_split, dummy_tg, dummy_og)
print(f'\n  Dummy 1-1 AW-MAE: {awmae_dummy:.4f}')
print(f'  Improvement vs dummy: {(awmae_dummy - awmae_val) / awmae_dummy * 100:.1f}%')


=== Validation Results ===
  AW-MAE                   : 2.8816
  MAE (raw)                : 0.9588
  Exact %                  : 11.7357
  Outcome % correct        : 52.2732
  GD % correct             : 23.9306
  Avg penalty              : 0.4982
  Avg loss                 : 2.8921

  Dummy 1-1 AW-MAE: 4.4959
  Improvement vs dummy: 35.9%


In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def plot_feature_importance(model, features, title='Feature Importance', top_n=30):
    fi = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=True).tail(top_n)
    
    fig, ax = plt.subplots(figsize=(8, top_n * 0.3))
    ax.barh(fi['feature'], fi['importance'])
    ax.set_title(title)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
    plt.close()
    print('Saved feature_importance.png')
    return fi

fi_df = plot_feature_importance(model_tg_val, ALL_FEATURES, 'Feature Importance — team_goals')
print('\nTop 15 features:')
print(fi_df.tail(15)[['feature', 'importance']].to_string())

Saved feature_importance.png

Top 15 features:
                         feature  importance
5             confederation_team    1.608998
6              confederation_opp    1.839375
16           distance_travel_opp    2.163221
23                elo_team_recon    2.802991
24            elo_opponent_recon    2.976608
10                population_opp    3.576042
3                     tournament    3.613897
34            opp_gd_last5_recon    4.449434
0                         gender    4.729308
1                           team    6.044905
29  opp_avg_conceded_last5_recon    6.273579
7                        is_home    6.305167
2                       opponent    8.187007
35                 gd_diff_recon    9.854404
25                elo_diff_recon   16.242592


In [15]:
# ── Retrain pada seluruh data modern untuk submission ──
# ⚠️  Jangan lakukan ini kalau val score belum memuaskan

print(f'Val AW-MAE: {awmae_val:.4f}')
print(f'Dummy AW-MAE: {awmae_dummy:.4f}')

RETRAIN = awmae_val < awmae_dummy  # hanya retrain kalau beat dummy
print(f'Retrain untuk submission: {RETRAIN}')

Val AW-MAE: 2.8816
Dummy AW-MAE: 4.4959
Retrain untuk submission: True


In [16]:
if RETRAIN:
    # Gunakan seluruh train modern sebagai training, tanpa validation set
    # Pakai best iteration dari val experiment sebagai n_estimators
    best_iter_tg = model_tg_val.best_iteration_
    best_iter_og = model_og_val.best_iteration_
    print(f'Best iterations: team_goals={best_iter_tg}, opp_goals={best_iter_og}')
    
    full_pool_tg = make_pool(train_modern, ALL_FEATURES, CAT_FEATURES, 'team_goals')
    full_pool_og = make_pool(train_modern, ALL_FEATURES, CAT_FEATURES, 'opp_goals')
    
    model_tg_final = CatBoostRegressor(
        iterations=best_iter_tg + 50,  # sedikit lebih banyak karena data lebih besar
        learning_rate=0.05,
        depth=6,
        loss_function='MAE',
        random_seed=RANDOM_SEED,
        verbose=200
    )
    model_og_final = CatBoostRegressor(
        iterations=best_iter_og + 50,
        learning_rate=0.05,
        depth=6,
        loss_function='MAE',
        random_seed=RANDOM_SEED,
        verbose=200
    )
    
    print('Training final model (team_goals)...')
    model_tg_final.fit(full_pool_tg)
    print('Training final model (opp_goals)...')
    model_og_final.fit(full_pool_og)
    print('Final models trained.')
else:
    print('⚠️  Val score tidak beat dummy — cek pipeline sebelum submit!')
    model_tg_final = model_tg_val
    model_og_final = model_og_val

Best iterations: team_goals=518, opp_goals=490
Training final model (team_goals)...
0:	learn: 1.1463761	total: 63.4ms	remaining: 35.9s
200:	learn: 0.9855702	total: 11.8s	remaining: 21.5s
400:	learn: 0.9608156	total: 20.6s	remaining: 8.6s
567:	learn: 0.9481615	total: 26.5s	remaining: 0us
Training final model (opp_goals)...
0:	learn: 1.1462133	total: 33.9ms	remaining: 18.3s
200:	learn: 0.9843814	total: 6.98s	remaining: 11.8s
400:	learn: 0.9592269	total: 14s	remaining: 4.85s
539:	learn: 0.9479912	total: 18.9s	remaining: 0us
Final models trained.


In [17]:
# Prediksi test
test_pool_tg = make_pool(test_fe, ALL_FEATURES, CAT_FEATURES)
test_pool_og = make_pool(test_fe, ALL_FEATURES, CAT_FEATURES)

pred_test_tg = model_tg_final.predict(test_pool_tg).clip(0)
pred_test_og = model_og_final.predict(test_pool_og).clip(0)

# Sanity check distribusi prediksi
print('Distribusi prediksi test:')
print(f'  team_goals: mean={pred_test_tg.mean():.3f}, std={pred_test_tg.std():.3f}, '
      f'min={pred_test_tg.min():.3f}, max={pred_test_tg.max():.3f}')
print(f'  opp_goals:  mean={pred_test_og.mean():.3f}, std={pred_test_og.std():.3f}, '
      f'min={pred_test_og.min():.3f}, max={pred_test_og.max():.3f}')

# Distribusi harus mirip dengan train target stats:
print(f'\nTrain target stats (referensi):')
print(f'  team_goals: mean={train_raw["team_goals"].mean():.3f}')
print(f'  opp_goals:  mean={train_raw["opp_goals"].mean():.3f}')

Distribusi prediksi test:
  team_goals: mean=1.222, std=0.792, min=0.000, max=7.787
  opp_goals:  mean=1.213, std=0.773, min=0.000, max=8.191

Train target stats (referensi):
  team_goals: mean=1.563
  opp_goals:  mean=1.563


In [18]:
# Build submission
submission = pd.DataFrame({
    'Id': test_fe['Id'].values,
    'team_goals': pred_test_tg,
    'opp_goals':  pred_test_og
})

# Verifikasi alignment dengan sample submission
assert set(submission['Id']) == set(sample['Id']), '❌ Id mismatch dengan sample submission!'
submission = submission.set_index('Id').loc[sample['Id']].reset_index()

submission.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Submission saved: {OUTPUT_PATH}')
print(f'   Shape: {submission.shape}')
print(submission.head(10))

✅ Submission saved: C:\Users\Gusti Jogish\Downloads\Gammafest\dataset\submission.csv
   Shape: (42422, 3)
                    Id  team_goals  opp_goals
0   M034984_Seychelles    1.181777   0.948687
1    M034984_Mauritius    1.087540   1.395419
2      M034985_Comoros    0.805795   1.493175
3     M034985_Maldives    1.376131   0.824762
4      M034986_Réunion    1.590582   0.684429
5   M034986_Madagascar    0.543765   1.485511
6  M034987_El Salvador    0.899170   1.453972
7    M034987_Venezuela    1.310638   0.900819
8      M034988_Mayotte    0.711640   1.684525
9      M034988_Réunion    1.734000   0.825375


In [19]:
def pre_submit_checklist(submission, sample, awmae_val, awmae_dummy, pred_test_tg, pred_test_og):
    checks = []
    
    # 1. Shape
    ok = submission.shape[0] == sample.shape[0]
    checks.append(('Row count match sample', ok, f'{submission.shape[0]} vs {sample.shape[0]}'))
    
    # 2. No NaN
    nan_tg = submission['team_goals'].isna().sum()
    nan_og = submission['opp_goals'].isna().sum()
    checks.append(('No NaN in predictions', nan_tg == 0 and nan_og == 0, f'NaN: tg={nan_tg}, og={nan_og}'))
    
    # 3. No negative
    neg = (pred_test_tg < 0).sum() + (pred_test_og < 0).sum()
    checks.append(('No negative predictions', neg == 0, f'Negative count: {neg}'))
    
    # 4. Beat dummy
    checks.append(('Val AW-MAE beats dummy', awmae_val < awmae_dummy,
                   f'Val={awmae_val:.4f} vs Dummy={awmae_dummy:.4f}'))
    
    # 5. Reasonable distribution
    mean_goals = (pred_test_tg.mean() + pred_test_og.mean()) / 2
    ok = 0.8 < mean_goals < 2.5
    checks.append(('Mean goals in sane range (0.8–2.5)', ok, f'Mean: {mean_goals:.3f}'))
    
    # 6. Id alignment
    id_ok = list(submission['Id']) == list(sample['Id'])
    checks.append(('Id order matches sample', id_ok, ''))
    
    print('=== PRE-SUBMIT CHECKLIST ===')
    all_pass = True
    for name, passed, note in checks:
        icon = '✅' if passed else '❌'
        print(f'  {icon} {name}' + (f' — {note}' if note else ''))
        if not passed: all_pass = False
    
    print()
    if all_pass:
        print('✅ Semua check passed. AMAN untuk submit.')
    else:
        print('❌ Ada check yang GAGAL. Jangan submit sebelum fix!')
    return all_pass

pre_submit_checklist(submission, sample, awmae_val, awmae_dummy, pred_test_tg, pred_test_og)

=== PRE-SUBMIT CHECKLIST ===
  ✅ Row count match sample — 42422 vs 42422
  ✅ No NaN in predictions — NaN: tg=0, og=0
  ✅ No negative predictions — Negative count: 0
  ✅ Val AW-MAE beats dummy — Val=2.8816 vs Dummy=4.4959
  ✅ Mean goals in sane range (0.8–2.5) — Mean: 1.217
  ✅ Id order matches sample

✅ Semua check passed. AMAN untuk submit.


True